# 07 Anomalies (BQ7): Python version

**BQ7:** Which days and which transactions are unusual?

- **Part A, unusual days.** Daily net revenue is compared with what is normal for that weekday and that month
  (regression residuals). Days are flagged by a z-score rule, an individuals control chart and IQR fences.
- **Part B, unusual invoices.** Each invoice is described by its size, value, prices and composition, and scored with an
  **Isolation Forest**; the most anomalous 1% are profiled. Univariate IQR fences are shown for comparison.

Period: 1 Dec 2009 – 30 Nov 2011 (complete months). Same definitions as `07_anomalies_r.ipynb`: the residuals,
z-scores, control limits and IQR fences must match exactly; Isolation Forest is randomised and implemented differently
in scikit-learn and R's `isotree`, so its scores are compared by rank correlation and overlap.

## Setup and data

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import IsolationForest

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
DASH = ROOT / "reports" / "dashboard_data"
FIG.mkdir(parents=True, exist_ok=True)
DASH.mkdir(parents=True, exist_ok=True)

START, END = "2009-12-01", "2011-12-01"
Z_LIMIT = 3.0          # |z| above this = unusual day
IQR_K = 3.0            # IQR fences at Q1 - 3*IQR and Q3 + 3*IQR ("extreme" outliers)
TOP_SHARE = 0.01       # Isolation Forest: flag the most anomalous 1% of invoices
SEED = 42

sales = pd.read_parquet(PROC / "sales_clean.parquet")
cancels = pd.read_parquet(PROC / "cancellations.parquet")
sales = sales.loc[(sales["invoice_date"] >= START) & (sales["invoice_date"] < END)].copy()
cancels = cancels.loc[(cancels["invoice_date"] >= START) & (cancels["invoice_date"] < END)].copy()
print(f"Sales lines: {len(sales):,}   cancellation lines: {len(cancels):,}")
print("Sales lines by type:", sales["line_type"].value_counts().to_dict())

# Part A: Unusual days

**Daily net revenue** = product sales minus product cancellations on each trading day (days with no sales are not
trading days and are not included). What is "normal" changes with the season, the trend and the weekday, so each day
is compared with a regression:

`log(net revenue) ~ weekday + calendar month (one level per year-month)`

The residual is the day's deviation from its expected level, in log terms (0.5 ≈ 65% above expected).
Three rules flag unusual days:

| Rule | Unusual if |
|---|---|
| **z-score** | the residual divided by its standard deviation is beyond ±3 |
| **Individuals control chart (I-MR)** | the residual is outside mean ± 2.66 × average moving range (the standard Shewhart limits for single observations) |
| **IQR fences** | the residual is below Q1 − 3·IQR or above Q3 + 3·IQR |

In [ ]:
prod = sales.loc[sales["line_type"] == "product"]
canc = cancels.loc[cancels["line_type"] == "product"]
daily = (prod.groupby("date")["revenue"].sum()
         .add(canc.groupby("date")["revenue"].sum(), fill_value=0)
         .rename("net").to_frame())
daily = daily.loc[daily.index.isin(prod["date"].unique())]        # trading days only
n_nonpos = int((daily["net"] <= 0).sum())
daily = daily.loc[daily["net"] > 0].sort_index()
daily["weekday"] = daily.index.day_name()
daily["year_month"] = daily.index.strftime("%Y-%m")

fit = smf.ols("np.log(net) ~ C(weekday) + C(year_month)", data=daily).fit()
daily["expected"] = np.exp(fit.fittedvalues)
daily["resid"] = fit.resid
daily["z"] = daily["resid"] / daily["resid"].std()

mr = daily["resid"].diff().abs()
cl, mr_bar = daily["resid"].mean(), mr.mean()
ucl, lcl = cl + 2.66 * mr_bar, cl - 2.66 * mr_bar
q1, q3 = daily["resid"].quantile([0.25, 0.75])
lo_f, hi_f = q1 - IQR_K * (q3 - q1), q3 + IQR_K * (q3 - q1)

daily["flag_z"] = daily["z"].abs() > Z_LIMIT
daily["flag_chart"] = (daily["resid"] > ucl) | (daily["resid"] < lcl)
daily["flag_iqr"] = (daily["resid"] > hi_f) | (daily["resid"] < lo_f)
daily["n_rules"] = daily[["flag_z", "flag_chart", "flag_iqr"]].sum(axis=1)

print(f"Trading days: {len(daily):,} (excluded {n_nonpos} with net revenue <= 0)   R² of the daily model: {fit.rsquared:.3f}")
print(f"Residual SD: {daily['resid'].std():.4f}   control limits: [{lcl:.4f}, {ucl:.4f}]   IQR fences: [{lo_f:.4f}, {hi_f:.4f}]")
print(f"Days flagged: z-score {daily['flag_z'].sum()}, control chart {daily['flag_chart'].sum()}, "
      f"IQR {daily['flag_iqr'].sum()}, by all three {(daily['n_rules'] == 3).sum()}, by at least one {(daily['n_rules'] >= 1).sum()}")

In [ ]:
show = daily.loc[daily["n_rules"] >= 1, ["weekday", "net", "expected", "resid", "z", "flag_z", "flag_chart", "flag_iqr"]]
show = show.reindex(show["z"].abs().sort_values(ascending=False).index)
print("Unusual days (largest deviation first):")
print(show.head(20).to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(daily.index, daily["net"] / 1000, color="grey", lw=0.7, label="Net revenue")
axes[0].plot(daily.index, daily["expected"] / 1000, color="#64B5F6", lw=1, label="Expected (weekday + month)")
f = daily.loc[daily["n_rules"] >= 1]
axes[0].scatter(f.index, f["net"] / 1000, color="#FF8A65", s=25, zorder=3, label="Flagged")
axes[0].set_ylabel("£k per day"); axes[0].set_title("Daily net revenue and unusual days"); axes[0].legend(fontsize=9)

axes[1].plot(daily.index, daily["resid"], color="grey", lw=0.7)
for y, ls in [(ucl, "--"), (lcl, "--"), (cl, "-")]:
    axes[1].axhline(y, color="#FF8A65" if ls == "--" else "#64B5F6", ls=ls, lw=1)
fc = daily.loc[daily["flag_chart"]]
axes[1].scatter(fc.index, fc["resid"], color="#FF8A65", s=20, zorder=3)
axes[1].set_ylabel("Residual (log)"); axes[1].set_title("Individuals control chart of residuals (limits: mean ± 2.66 × average moving range)")
fig.tight_layout(); fig.savefig(FIG / "bq7_unusual_days.png", dpi=150); plt.show()

# Part B: Unusual invoices

All sales invoices in the period, including non-product lines (postage, fees, adjustments), because the hypothesis
is that unusual cases concentrate in large quantities, cancellations and non-product codes.

| Feature (per invoice) | Definition |
|---|---|
| `log_lines` | log(number of lines) |
| `log_qty` | log(1 + total quantity) |
| `log_value` | log(1 + invoice value, £) |
| `log_avg_price` | log(value ÷ quantity) |
| `nonproduct_share` | share of the value from non-product lines |
| `max_price_dev` | largest absolute log price ratio on the invoice (how far a price is from the product's normal price) |
| `cancelled_share` | share of the value later cancelled with an exact match |
| `has_customer`, `is_uk` | 1 / 0 |
| `hour` | hour of the invoice |

**Isolation Forest** isolates each invoice by random splits; unusual invoices are separated after few splits.
It needs no labels and looks at all features together, so it finds combinations that no single threshold would catch.

In [ ]:
s = sales.copy()
s["nonprod_rev"] = s["revenue"].where(s["line_type"] != "product", 0.0)
s["cancel_rev"] = s["revenue"].where(s["cancelled_exact"], 0.0)
s["price_dev"] = np.abs(np.log(s["price_ratio"])).where(s["line_type"] == "product")
inv = s.groupby("invoice").agg(lines=("stock_code", "size"), qty=("quantity", "sum"), value=("revenue", "sum"),
                               nonprod_rev=("nonprod_rev", "sum"), cancel_rev=("cancel_rev", "sum"),
                               max_price_dev=("price_dev", "max"), has_customer=("has_customer", "max"),
                               is_uk=("is_uk", "max"), hour=("hour", "min"), any_bulk=("is_bulk", "max"),
                               date=("date", "min")).sort_index()
inv["max_price_dev"] = inv["max_price_dev"].fillna(0)
inv["nonproduct_share"] = inv["nonprod_rev"] / inv["value"]
inv["cancelled_share"] = inv["cancel_rev"] / inv["value"]
X = pd.DataFrame({"log_lines": np.log(inv["lines"]), "log_qty": np.log1p(inv["qty"]),
                  "log_value": np.log1p(inv["value"]), "log_avg_price": np.log(inv["value"] / inv["qty"]),
                  "nonproduct_share": inv["nonproduct_share"], "max_price_dev": inv["max_price_dev"],
                  "cancelled_share": inv["cancelled_share"], "has_customer": inv["has_customer"].astype(int),
                  "is_uk": inv["is_uk"].astype(int), "hour": inv["hour"].astype(float)}, index=inv.index)
print(f"Invoices: {len(inv):,}")
X.describe().T

In [ ]:
iso = IsolationForest(n_estimators=200, max_samples=256, random_state=SEED).fit(X.values)
inv["score"] = -iso.score_samples(X.values)                 # higher = more anomalous
n_top = int(np.ceil(TOP_SHARE * len(inv)))
inv["flag_iforest"] = inv["score"].rank(method="first", ascending=False) <= n_top

def iqr_flag(x):
    q1, q3 = x.quantile([0.25, 0.75])
    return (x < q1 - IQR_K * (q3 - q1)) | (x > q3 + IQR_K * (q3 - q1))
inv["flag_iqr_value"] = iqr_flag(X["log_value"])
inv["flag_iqr_qty"] = iqr_flag(X["log_qty"])

print(f"Isolation Forest: top {TOP_SHARE:.0%} = {n_top} invoices flagged")
print(f"IQR fences (3 x IQR): value {inv['flag_iqr_value'].sum()}, quantity {inv['flag_iqr_qty'].sum()}; "
      f"also flagged by Isolation Forest: {(inv['flag_iforest'] & (inv['flag_iqr_value'] | inv['flag_iqr_qty'])).sum()}")

profile = pd.DataFrame({
    "invoices": [inv["flag_iforest"].sum(), (~inv["flag_iforest"]).sum()],
    "median_value": [inv.loc[inv["flag_iforest"], "value"].median(), inv.loc[~inv["flag_iforest"], "value"].median()],
    "median_lines": [inv.loc[inv["flag_iforest"], "lines"].median(), inv.loc[~inv["flag_iforest"], "lines"].median()],
    **{name: [cond[inv["flag_iforest"]].mean(), cond[~inv["flag_iforest"]].mean()] for name, cond in {
        "has_nonproduct": inv["nonprod_rev"] > 0, "has_cancelled": inv["cancel_rev"] > 0,
        "has_bulk_line": inv["any_bulk"].astype(bool), "has_price_outlier": inv["max_price_dev"] > np.log(5),
        "no_customer": ~inv["has_customer"].astype(bool), "non_uk": ~inv["is_uk"].astype(bool)}.items()}},
    index=["Flagged (top 1%)", "All other invoices"])
print("\nProfile of flagged vs other invoices (shares of invoices):")
profile.T

In [ ]:
cols = ["date", "lines", "qty", "value", "nonproduct_share", "max_price_dev", "cancelled_share", "has_customer", "is_uk", "hour", "score"]
print("Ten most anomalous invoices:")
print(inv.sort_values(["score"], ascending=False)[cols].head(10).to_string())

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(X["log_qty"], X["log_value"], s=3, color="grey", alpha=0.3, label="Invoices")
fl = inv["flag_iforest"]
ax.scatter(X.loc[fl, "log_qty"], X.loc[fl, "log_value"], s=12, color="#FF8A65", label=f"Isolation Forest top {TOP_SHARE:.0%}")
ax.set_xlabel("log(1 + total quantity)"); ax.set_ylabel("log(1 + invoice value)")
ax.set_title("Invoices: size vs value, with the most anomalous highlighted"); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(FIG / "bq7_unusual_invoices.png", dpi=150); plt.show()

## Export for the dashboard

In [ ]:
days_out = daily.loc[daily["n_rules"] >= 1, ["weekday", "net", "expected", "resid", "z", "flag_z", "flag_chart", "flag_iqr"]]
days_out.round(4).to_csv(DASH / "anomaly_days.csv")
daily[["net", "expected", "resid", "z"]].round(4).to_csv(DASH / "anomaly_daily_residuals.csv")
inv[["score"]].round(6).to_csv(DASH / "anomaly_invoice_scores.csv")
inv.loc[inv["flag_iforest"], cols].round(4).to_csv(DASH / "anomaly_invoices_top.csv")
profile.round(4).to_csv(DASH / "anomaly_profile.csv")
for f in ["anomaly_days.csv", "anomaly_daily_residuals.csv", "anomaly_invoice_scores.csv",
          "anomaly_invoices_top.csv", "anomaly_profile.csv"]:
    print(f"{f:<30} {len(pd.read_csv(DASH / f)):>6,} rows")